# Dataset v1 sampling and review

This notebook prepares a **200-image BDD100K dataset** for a licence plate detector whose target is plates containing human-readable registration information. It preserves the official train, validation, and test boundaries and does not perform annotation or model training.

The workflow is deliberately split into explicit stages:

1. extract image-level vehicle statistics from the official train/validation annotations;
2. optionally run cached COCO YOLO11n vehicle inference on a deterministic subset of the unannotated test split;
3. build append-only candidate pools;
4. manually review candidates with decisions saved after every click; and
5. validate and materialize the final dataset only after every quota is met.

All generated files live under `data/dataset_v1/`. Raw BDD100K images are read in place and are never modified.

## 1. Imports and configuration

Configuration is kept together so the sampling seed, pool sizes, area bands, and review quotas are easy to inspect. Paths are resolved from the repository root rather than from a machine-specific absolute path.

The test inference subset is larger than the final 100-image test candidate pool so the same area-based priorities can be applied after COCO detections are cached. Increase `TEST_INFERENCE_SUBSET_SIZE` or a pool target later to expand the pool; existing candidates and review decisions will be retained.

In [1]:
from __future__ import annotations

from io import BytesIO
from pathlib import Path
import hashlib
import html
import json
import shutil

import pandas as pd
from PIL import Image
from IPython.display import display


SEED = 42
VEHICLE_CATEGORIES = {"car", "truck", "bus"}

CANDIDATE_POOL_TARGETS = {
    "train": 1500,
    "val": 300,
    "test": 300,
}

# Ten percent of each candidate pool is generated separately from images
# with no detected/annotated cars, trucks, or buses.
NEGATIVE_CANDIDATE_FRACTION = 0.10

AREA_BANDS = {
    "30k_60k": (30_000, 60_000, 0.10),
    "60k_150k": (60_000, 150_000, 0.35),
    "150k_300k": (150_000, 300_000, 0.40),
    "above_300k": (300_000, None, 0.15),
}

QUOTAS = {
    "train": {"clear": 128, "challenging": 16, "negative": 16},
    "val": {"clear": 16, "challenging": 2, "negative": 2},
    "test": {"clear": 16, "challenging": 2, "negative": 2},
}

TEST_INFERENCE_SUBSET_SIZE = 1_000
TEST_INFERENCE_BATCH_SIZE = 8
TEST_INFERENCE_CONFIDENCE = 0.25
TEST_CACHE_FLUSH_EVERY = 25


def find_project_root(start: Path | None = None) -> Path:
    "Find the repository root without assuming how Jupyter was launched."
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "PROJECT_PLAN.md").exists():
            return candidate
    raise FileNotFoundError("Could not find PROJECT_PLAN.md above the current directory.")


PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "bdd100k"
OUTPUT_DIR = PROJECT_ROOT / "data" / "dataset_v1"

SPLIT_IMAGE_DIRS = {
    "train": DATASET_ROOT / "train" / "images",
    "val": DATASET_ROOT / "val" / "images",
    "test": DATASET_ROOT / "test",
}
ANNOTATION_FILES = {
    "train": DATASET_ROOT / "train" / "annotations" / "bdd100k_labels_images_train.json",
    "val": DATASET_ROOT / "val" / "annotations" / "bdd100k_labels_images_val.json",
}

CANDIDATES_PATH = OUTPUT_DIR / "candidates.csv"
REVIEW_PROGRESS_PATH = OUTPUT_DIR / "review_progress.csv"
MANIFEST_PATH = OUTPUT_DIR / "dataset_manifest.csv"
TEST_STATS_CACHE_PATH = OUTPUT_DIR / "test_vehicle_stats.csv"

for split, image_dir in SPLIT_IMAGE_DIRS.items():
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Missing {split} image directory: {image_dir}")
for split, annotation_path in ANNOTATION_FILES.items():
    if not annotation_path.is_file():
        raise FileNotFoundError(f"Missing {split} annotations: {annotation_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\aarish.tickoo\workspace\traffic-pii-masking
Output directory: data\dataset_v1


## 2. Extract train and validation vehicle statistics

BDD100K supplies object annotations for train and validation. The code below reuses the POC idea of summarizing vehicle boxes at image level, but also records the largest vehicle's width, height, coordinates, vehicle count, metadata, and source path. Only `car`, `truck`, and `bus` boxes are used, matching this experiment's candidate-generation definition.

The largest-box coordinates are retained so the review widget can optionally display a close crop without changing the original image.

In [2]:
def project_relative(path: Path) -> str:
    "Return a portable path relative to the repository root."
    return path.resolve().relative_to(PROJECT_ROOT).as_posix()


def load_bdd_annotations(path: Path) -> list[dict]:
    try:
        with path.open("r", encoding="utf-8") as file:
            records = json.load(file)
    except (OSError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"Could not read BDD100K annotations from {path}") from exc
    if not isinstance(records, list):
        raise ValueError(f"Expected a list of annotation records in {path}")
    return records


def annotation_vehicle_stats(split: str, annotations: list[dict]) -> pd.DataFrame:
    "Convert BDD100K image annotations into one row per image."
    rows: list[dict] = []
    image_dir = SPLIT_IMAGE_DIRS[split]

    for record in annotations:
        largest: dict | None = None
        vehicle_count = 0

        for label in record.get("labels", []):
            if label.get("category") not in VEHICLE_CATEGORIES:
                continue
            box = label.get("box2d")
            if not box:
                continue

            width = float(box["x2"]) - float(box["x1"])
            height = float(box["y2"]) - float(box["y1"])
            if width <= 0 or height <= 0:
                continue

            vehicle_count += 1
            area = width * height
            if largest is None or area > largest["area"]:
                largest = {
                    "area": area,
                    "width": width,
                    "height": height,
                    "x1": float(box["x1"]),
                    "y1": float(box["y1"]),
                    "x2": float(box["x2"]),
                    "y2": float(box["y2"]),
                }

        image_id = record["name"]
        attributes = record.get("attributes", {})
        source_path = image_dir / image_id
        rows.append(
            {
                "image_id": image_id,
                "split": split,
                "vehicle_count": vehicle_count,
                "max_vehicle_area": 0.0 if largest is None else largest["area"],
                "max_vehicle_width": 0.0 if largest is None else largest["width"],
                "max_vehicle_height": 0.0 if largest is None else largest["height"],
                "max_vehicle_x1": pd.NA if largest is None else largest["x1"],
                "max_vehicle_y1": pd.NA if largest is None else largest["y1"],
                "max_vehicle_x2": pd.NA if largest is None else largest["x2"],
                "max_vehicle_y2": pd.NA if largest is None else largest["y2"],
                "timeofday": attributes.get("timeofday"),
                "weather": attributes.get("weather"),
                "scene": attributes.get("scene"),
                "source_path": project_relative(source_path),
                "stats_source": "bdd100k_annotations",
            }
        )

    stats = pd.DataFrame(rows)
    if stats.duplicated(["split", "image_id"]).any():
        raise ValueError(f"Duplicate image IDs found in {split} annotations.")
    return stats


train_annotations = load_bdd_annotations(ANNOTATION_FILES["train"])
val_annotations = load_bdd_annotations(ANNOTATION_FILES["val"])
train_stats = annotation_vehicle_stats("train", train_annotations)
val_stats = annotation_vehicle_stats("val", val_annotations)

annotated_stats = pd.concat([train_stats, val_stats], ignore_index=True)
missing_sources = annotated_stats.loc[
    ~annotated_stats["source_path"].map(lambda value: (PROJECT_ROOT / value).is_file())
]
if not missing_sources.empty:
    raise FileNotFoundError(
        f"{len(missing_sources)} annotated source images are missing; first: "
        f"{missing_sources.iloc[0]['source_path']}"
    )

annotated_stats.groupby("split").agg(
    images=("image_id", "size"),
    images_with_vehicles=("vehicle_count", lambda values: int((values > 0).sum())),
    median_max_vehicle_area=("max_vehicle_area", "median"),
)

,images,images_with_vehicles,median_max_vehicle_area
split,,,
train,69863,69257,38785.247342
val,10000,9904,38296.365938


## 3. Cache test-split COCO vehicle detections

The official BDD100K test split has no object annotation JSON. To obtain comparable vehicle statistics, this stage takes a deterministic random subset and runs pretrained COCO YOLO11n for car, bus, and truck detections on CPU.

This is the only expensive stage. It is **not launched automatically**: set `RUN_TEST_INFERENCE = True` in the execution cell when ready. Results are appended to `test_vehicle_stats.csv` every 25 images, and already cached image IDs are skipped on later runs. The pretrained detector is used only for sampling metadata, not for licence plate labels or model evaluation.

In [3]:
TEST_COCO_CLASS_IDS = [2, 5, 7]  # car, bus, truck


def deterministic_test_subset(size: int = TEST_INFERENCE_SUBSET_SIZE) -> list[Path]:
    images = sorted(SPLIT_IMAGE_DIRS["test"].glob("*.jpg"))
    if size > len(images):
        raise ValueError(f"Requested {size} test images, but only {len(images)} are available.")
    ranked = sorted(
        images,
        key=lambda path: hashlib.sha256(
            f"{SEED}|test-inference|{path.name}".encode("utf-8")
        ).hexdigest(),
    )
    return ranked[:size]


def atomic_write_csv(frame: pd.DataFrame, destination: Path) -> None:
    "Write a CSV via a temporary file so an interrupted write keeps the prior file."
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(destination)


def yolo_result_to_stats(result, image_path: Path) -> dict:
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        largest = None
        vehicle_count = 0
    else:
        xyxy = boxes.xyxy.detach().cpu()
        widths = xyxy[:, 2] - xyxy[:, 0]
        heights = xyxy[:, 3] - xyxy[:, 1]
        areas = widths * heights
        largest_index = int(areas.argmax().item())
        largest = {
            "area": float(areas[largest_index].item()),
            "width": float(widths[largest_index].item()),
            "height": float(heights[largest_index].item()),
            "xyxy": [float(value) for value in xyxy[largest_index].tolist()],
        }
        vehicle_count = len(xyxy)

    coordinates = [pd.NA] * 4 if largest is None else largest["xyxy"]
    return {
        "image_id": image_path.name,
        "split": "test",
        "vehicle_count": vehicle_count,
        "max_vehicle_area": 0.0 if largest is None else largest["area"],
        "max_vehicle_width": 0.0 if largest is None else largest["width"],
        "max_vehicle_height": 0.0 if largest is None else largest["height"],
        "max_vehicle_x1": coordinates[0],
        "max_vehicle_y1": coordinates[1],
        "max_vehicle_x2": coordinates[2],
        "max_vehicle_y2": coordinates[3],
        "timeofday": pd.NA,
        "weather": pd.NA,
        "scene": pd.NA,
        "source_path": project_relative(image_path),
        "stats_source": "coco_yolo11n_cpu",
    }


def generate_test_vehicle_stats() -> pd.DataFrame:
    "Run missing test detections and checkpoint the cache incrementally."
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise ImportError(
            "Install ultralytics in this notebook environment before test inference."
        ) from exc

    subset = deterministic_test_subset()
    if TEST_STATS_CACHE_PATH.exists():
        cached = pd.read_csv(TEST_STATS_CACHE_PATH)
    else:
        cached = pd.DataFrame()

    cached_ids = set(cached.get("image_id", pd.Series(dtype="string")).astype(str))
    remaining = [path for path in subset if path.name not in cached_ids]
    print(f"Test subset: {len(subset)} | cached: {len(cached_ids)} | remaining: {len(remaining)}")
    if not remaining:
        return cached.sort_values("image_id").reset_index(drop=True)

    model = YOLO("yolo11n.pt")
    new_rows: list[dict] = []

    for batch_start in range(0, len(remaining), TEST_INFERENCE_BATCH_SIZE):
        batch_paths = remaining[batch_start : batch_start + TEST_INFERENCE_BATCH_SIZE]
        results = model.predict(
            source=[str(path) for path in batch_paths],
            device="cpu",
            classes=TEST_COCO_CLASS_IDS,
            conf=TEST_INFERENCE_CONFIDENCE,
            verbose=False,
        )
        new_rows.extend(
            yolo_result_to_stats(result, path)
            for result, path in zip(results, batch_paths, strict=True)
        )

        processed = batch_start + len(batch_paths)
        if processed % TEST_CACHE_FLUSH_EVERY < TEST_INFERENCE_BATCH_SIZE or processed == len(remaining):
            combined = pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True)
            combined = combined.drop_duplicates("image_id", keep="last")
            combined = combined.sort_values("image_id").reset_index(drop=True)
            atomic_write_csv(combined, TEST_STATS_CACHE_PATH)
            print(f"Cached {len(combined)} test rows ({processed}/{len(remaining)} new images processed).")

    return pd.read_csv(TEST_STATS_CACHE_PATH)

In [4]:
# Explicit opt-in: change to True only when you are ready for CPU inference.
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE:
    test_stats = generate_test_vehicle_stats()
elif TEST_STATS_CACHE_PATH.exists():
    test_stats = pd.read_csv(TEST_STATS_CACHE_PATH)
    print(f"Loaded {len(test_stats)} cached test rows.")
else:
    test_stats = pd.DataFrame()
    print(
        "No test cache exists yet. Set RUN_TEST_INFERENCE = True and rerun this cell "
        "before generating all three candidate pools."
    )

Test subset: 1000 | cached: 0 | remaining: 1000
Cached 32 test rows (32/1000 new images processed).
Cached 56 test rows (56/1000 new images processed).
Cached 80 test rows (80/1000 new images processed).
Cached 104 test rows (104/1000 new images processed).
Cached 128 test rows (128/1000 new images processed).
Cached 152 test rows (152/1000 new images processed).
Cached 176 test rows (176/1000 new images processed).
Cached 200 test rows (200/1000 new images processed).
Cached 232 test rows (232/1000 new images processed).
Cached 256 test rows (256/1000 new images processed).
Cached 280 test rows (280/1000 new images processed).
Cached 304 test rows (304/1000 new images processed).
Cached 328 test rows (328/1000 new images processed).
Cached 352 test rows (352/1000 new images processed).
Cached 376 test rows (376/1000 new images processed).
Cached 400 test rows (400/1000 new images processed).
Cached 432 test rows (432/1000 new images processed).
Cached 456 test rows (456/1000 new image

## 4. Generate append-only candidate pools

Positive candidates are selected from the requested maximum-vehicle-area bands using the 10% / 35% / 40% / 15% priority mix. Negative candidates are sampled separately from rows with no annotated or detected car, truck, or bus. A stable SHA-256 score based on seed 42 makes selection reproducible.

`candidates.csv` is append-only by `(split, image_id)`: increasing a target or test inference subset adds new candidates without removing old ones. `review_progress.csv` is merged the same way, so rerunning this section never resets an existing decision. The candidate label describes how an image entered the pool; the reviewer still makes the definitive clear/challenging/negative decision by inspecting the image.

In [5]:
def stable_score(split: str, image_id: str, purpose: str) -> str:
    return hashlib.sha256(
        f"{SEED}|{split}|{purpose}|{image_id}".encode("utf-8")
    ).hexdigest()


def allocate_counts(total: int) -> dict[str, int]:
    "Use largest remainders so rounded area-band counts sum exactly to total."
    raw = {name: total * values[2] for name, values in AREA_BANDS.items()}
    counts = {name: int(value) for name, value in raw.items()}
    remainder = total - sum(counts.values())
    order = sorted(raw, key=lambda name: raw[name] - counts[name], reverse=True)
    for name in order[:remainder]:
        counts[name] += 1
    return counts


def area_band_for(area: float) -> str | None:
    for name, (lower, upper, _) in AREA_BANDS.items():
        if area >= lower and (upper is None or area < upper):
            return name
    return None


def take_by_stable_score(frame: pd.DataFrame, count: int, purpose: str) -> pd.DataFrame:
    if count <= 0 or frame.empty:
        return frame.head(0).copy()
    ranked = frame.assign(
        _score=[stable_score(row.split, row.image_id, purpose) for row in frame.itertuples()]
    ).sort_values(["_score", "image_id"])
    return ranked.head(count).drop(columns="_score")


def sample_split_candidates(stats: pd.DataFrame, split: str, target: int) -> pd.DataFrame:
    split_stats = stats.loc[stats["split"] == split].copy()
    if split_stats.empty:
        raise ValueError(f"No vehicle statistics are available for split '{split}'.")

    negative_target = round(target * NEGATIVE_CANDIDATE_FRACTION)
    positive_target = target - negative_target
    split_stats["area_band"] = split_stats["max_vehicle_area"].map(area_band_for)

    negative_pool = split_stats.loc[split_stats["vehicle_count"] == 0].copy()
    negatives = take_by_stable_score(negative_pool, negative_target, "negative")
    negatives["candidate_type"] = "negative"
    negatives["area_band"] = "no_vehicle"

    positive_pool = split_stats.loc[
        (split_stats["vehicle_count"] > 0) & split_stats["area_band"].notna()
    ].copy()
    positive_parts: list[pd.DataFrame] = []
    for band_name, band_target in allocate_counts(positive_target).items():
        band_pool = positive_pool.loc[positive_pool["area_band"] == band_name]
        chosen = take_by_stable_score(band_pool, band_target, f"positive-{band_name}")
        positive_parts.append(chosen)

    positives = pd.concat(positive_parts, ignore_index=True)
    shortage = positive_target - len(positives)
    if shortage > 0:
        remaining = positive_pool.loc[~positive_pool["image_id"].isin(positives["image_id"])]
        fill = take_by_stable_score(remaining, shortage, "positive-shortage-fill")
        positives = pd.concat([positives, fill], ignore_index=True)
    positives["candidate_type"] = "positive"

    sampled = pd.concat([positives, negatives], ignore_index=True)
    if len(sampled) < target:
        raise ValueError(
            f"Only {len(sampled)} of {target} requested {split} candidates were available. "
            "Expand the test inference subset or inspect the area-band/negative supply."
        )
    sampled["sampling_seed"] = SEED
    sampled["candidate_score"] = [
        stable_score(row.split, row.image_id, "candidate-order")
        for row in sampled.itertuples()
    ]
    return sampled.sort_values(["candidate_score", "image_id"]).reset_index(drop=True)


def initialize_review_progress(candidates: pd.DataFrame) -> pd.DataFrame:
    base = candidates[["split", "image_id"]].copy()
    if REVIEW_PROGRESS_PATH.exists():
        existing = pd.read_csv(REVIEW_PROGRESS_PATH, dtype="string")
        required = {"split", "image_id", "decision"}
        if not required.issubset(existing.columns):
            raise ValueError(f"Existing {REVIEW_PROGRESS_PATH.name} has an incompatible schema.")
        progress = base.merge(existing, on=["split", "image_id"], how="left")
    else:
        progress = base
        progress["decision"] = pd.NA
        progress["reviewed_at_utc"] = pd.NA

    progress["decision"] = progress["decision"].fillna("unreviewed")
    if "reviewed_at_utc" not in progress:
        progress["reviewed_at_utc"] = pd.NA
    atomic_write_csv(progress, REVIEW_PROGRESS_PATH)
    return progress


def merge_candidate_pool(new_candidates: pd.DataFrame) -> pd.DataFrame:
    if CANDIDATES_PATH.exists():
        existing = pd.read_csv(CANDIDATES_PATH)
        required = {"split", "image_id", "candidate_order"}
        if not required.issubset(existing.columns):
            raise ValueError(f"Existing {CANDIDATES_PATH.name} has an incompatible schema.")
        unseen = new_candidates.merge(
            existing[["split", "image_id"]],
            on=["split", "image_id"],
            how="left",
            indicator=True,
        ).loc[lambda frame: frame["_merge"] == "left_only"].drop(columns="_merge")
        next_order = int(existing["candidate_order"].max()) + 1 if not existing.empty else 0
        unseen["candidate_order"] = range(next_order, next_order + len(unseen))
        combined = pd.concat([existing, unseen], ignore_index=True)
    else:
        combined = new_candidates.copy()
        combined["candidate_order"] = range(len(combined))

    if combined.duplicated(["split", "image_id"]).any():
        raise ValueError("Duplicate (split, image_id) pairs found in the candidate pool.")
    if combined["image_id"].duplicated().any():
        raise ValueError("An image ID appears in more than one candidate split.")
    combined = combined.sort_values("candidate_order").reset_index(drop=True)
    atomic_write_csv(combined, CANDIDATES_PATH)
    initialize_review_progress(combined)
    return combined

In [6]:
# This cell is inexpensive once test statistics are cached.
if test_stats.empty:
    raise RuntimeError(
        "Test statistics are not available. Complete the explicit test-inference stage first."
    )

all_stats = pd.concat([annotated_stats, test_stats], ignore_index=True)
requested_candidates = pd.concat(
    [
        sample_split_candidates(all_stats, split, target)
        for split, target in CANDIDATE_POOL_TARGETS.items()
    ],
    ignore_index=True,
)
candidates = merge_candidate_pool(requested_candidates)

print(f"Saved {len(candidates)} total candidates to {CANDIDATES_PATH.relative_to(PROJECT_ROOT)}")
display(
    candidates.groupby(["split", "candidate_type", "area_band"], dropna=False)
    .size()
    .rename("count")
    .to_frame()
)

Saved 2100 total candidates to data\dataset_v1\candidates.csv


count
split candidate_type area_band        
test  negative       no_vehicle     30
      positive       150k_300k     110
                     30k_60k        32
                     60k_150k       95
                     above_300k     33
train negative       no_vehicle    150
      positive       150k_300k     540
                     30k_60k       135
                     60k_150k      472
                     above_300k    203
val   negative       no_vehicle     30
      positive       150k_300k     108
                     30k_60k        27
                     60k_150k       95
                     above_300k     40

## 5. Persistent interactive review

The widget reloads candidates and review progress from disk each time it is launched. Every Clear, Challenging, Negative, Reject, or Skip click is written immediately. The three acceptance buttons are disabled independently when their quota for the selected split is full.

Use the split dropdown to move between official splits. The main image is displayed at a wide resolution for reading plates; enable **Show largest vehicle crop** for a closer look at the largest annotated/detected vehicle. **Previous** navigates without changing a decision, while **Undo** restores the most recent decision made during the current widget session and persists that restoration.

In [7]:
def launch_review_widget() -> None:
    "Create and display the persistent image-review interface."
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError as exc:
        raise ImportError("Install ipywidgets in the active notebook environment.") from exc

    if not CANDIDATES_PATH.exists() or not REVIEW_PROGRESS_PATH.exists():
        raise FileNotFoundError("Generate candidates and review_progress.csv before launching review.")

    candidate_frame = pd.read_csv(CANDIDATES_PATH).sort_values("candidate_order")
    progress_frame = pd.read_csv(REVIEW_PROGRESS_PATH, dtype="string")
    decision_map = {
        (row.split, row.image_id): row.decision
        for row in progress_frame.itertuples(index=False)
    }
    reviewed_at_map = {
        (row.split, row.image_id): getattr(row, "reviewed_at_utc", pd.NA)
        for row in progress_frame.itertuples(index=False)
    }
    undo_stack: list[tuple[str, str, str, object]] = []
    state = {"split": "train", "index": 0}

    split_dropdown = widgets.Dropdown(
        options=[("Train", "train"), ("Validation", "val"), ("Test", "test")],
        value="train",
        description="Split:",
    )
    image_widget = widgets.Image(format="jpeg", layout=widgets.Layout(width="1200px"))
    crop_widget = widgets.Image(format="jpeg", layout=widgets.Layout(width="700px"))
    crop_toggle = widgets.Checkbox(value=False, description="Show largest vehicle crop")
    status = widgets.HTML()
    metadata = widgets.HTML()
    progress = widgets.HTML()

    clear_button = widgets.Button(description="Clear", button_style="success")
    challenging_button = widgets.Button(description="Challenging", button_style="warning")
    negative_button = widgets.Button(description="Negative", button_style="info")
    reject_button = widgets.Button(description="Reject", button_style="danger")
    skip_button = widgets.Button(description="Skip")
    previous_button = widgets.Button(description="Previous")
    undo_button = widgets.Button(description="Undo")
    category_buttons = {
        "clear": clear_button,
        "challenging": challenging_button,
        "negative": negative_button,
    }

    def split_candidates() -> pd.DataFrame:
        return candidate_frame.loc[candidate_frame["split"] == state["split"]].reset_index(drop=True)

    def current_row() -> pd.Series:
        frame = split_candidates()
        if frame.empty:
            raise ValueError(f"No candidates exist for split {state['split']}.")
        state["index"] = max(0, min(state["index"], len(frame) - 1))
        return frame.iloc[state["index"]]

    def save_progress() -> None:
        rows = []
        for row in candidate_frame.itertuples(index=False):
            key = (row.split, row.image_id)
            rows.append(
                {
                    "split": row.split,
                    "image_id": row.image_id,
                    "decision": decision_map.get(key, "unreviewed"),
                    "reviewed_at_utc": reviewed_at_map.get(key, pd.NA),
                }
            )
        atomic_write_csv(pd.DataFrame(rows), REVIEW_PROGRESS_PATH)

    def counts_for_split(split: str) -> dict[str, int]:
        values = [decision for (row_split, _), decision in decision_map.items() if row_split == split]
        return {name: values.count(name) for name in ("clear", "challenging", "negative")}

    def jpeg_bytes(image: Image.Image, quality: int = 92) -> bytes:
        buffer = BytesIO()
        image.convert("RGB").save(buffer, format="JPEG", quality=quality)
        return buffer.getvalue()

    def update_crop(row: pd.Series, image: Image.Image) -> None:
        coordinates = [row.get(name) for name in (
            "max_vehicle_x1", "max_vehicle_y1", "max_vehicle_x2", "max_vehicle_y2"
        )]
        if not crop_toggle.value or any(pd.isna(value) for value in coordinates):
            crop_widget.layout.display = "none"
            crop_widget.value = b""
            return
        x1, y1, x2, y2 = [float(value) for value in coordinates]
        padding_x = max(20.0, (x2 - x1) * 0.10)
        padding_y = max(20.0, (y2 - y1) * 0.10)
        crop_box = (
            max(0, int(x1 - padding_x)),
            max(0, int(y1 - padding_y)),
            min(image.width, int(x2 + padding_x)),
            min(image.height, int(y2 + padding_y)),
        )
        crop_widget.value = jpeg_bytes(image.crop(crop_box))
        crop_widget.layout.display = "block"

    def render() -> None:
        frame = split_candidates()
        row = current_row()
        key = (row["split"], row["image_id"])
        source = PROJECT_ROOT / row["source_path"]
        if not source.is_file():
            raise FileNotFoundError(f"Candidate source image is missing: {source}")
        with Image.open(source) as image:
            image.load()
            image_widget.value = jpeg_bytes(image)
            update_crop(row, image)

        counts = counts_for_split(state["split"])
        values = [
            decision for (split, _), decision in decision_map.items() if split == state["split"]
        ]
        reviewed = sum(decision != "unreviewed" for decision in values)
        selected = sum(counts.values())
        current_decision = decision_map.get(key, "unreviewed")

        status.value = (
            f"<b>{html.escape(state['split'].upper())}</b> &nbsp; "
            f"Image {state['index'] + 1}/{len(frame)} &nbsp; | &nbsp; "
            f"Reviewed {reviewed}/{len(frame)} &nbsp; | &nbsp; "
            f"Selected {selected}/{sum(QUOTAS[state['split']].values())} &nbsp; | &nbsp; "
            f"Decision: <b>{html.escape(current_decision)}</b>"
        )
        progress.value = " &nbsp; | &nbsp; ".join(
            f"<b>{name.title()}:</b> {counts[name]}/{quota}"
            for name, quota in QUOTAS[state["split"]].items()
        )
        metadata.value = (
            f"<b>Image:</b> {html.escape(str(row['image_id']))}<br>"
            f"<b>Candidate type:</b> {html.escape(str(row['candidate_type']))} &nbsp; | &nbsp; "
            f"<b>Area band:</b> {html.escape(str(row['area_band']))}<br>"
            f"<b>Vehicles:</b> {int(row['vehicle_count'])} &nbsp; | &nbsp; "
            f"<b>Largest box:</b> {float(row['max_vehicle_width']):.0f} × "
            f"{float(row['max_vehicle_height']):.0f} px "
            f"({float(row['max_vehicle_area']):,.0f} px²)<br>"
            f"<b>Time:</b> {html.escape(str(row.get('timeofday', '')))} &nbsp; | &nbsp; "
            f"<b>Weather:</b> {html.escape(str(row.get('weather', '')))} &nbsp; | &nbsp; "
            f"<b>Scene:</b> {html.escape(str(row.get('scene', '')))}<br>"
            f"<b>Source:</b> {html.escape(str(row['source_path']))}"
        )

        for category, button in category_buttons.items():
            quota = QUOTAS[state["split"]][category]
            button.description = f"{category.title()} ({counts[category]}/{quota})"
            button.disabled = counts[category] >= quota and current_decision != category
        previous_button.disabled = state["index"] == 0
        undo_button.disabled = not undo_stack

    def move_next() -> None:
        frame = split_candidates()
        state["index"] = min(state["index"] + 1, len(frame) - 1)
        render()

    def record_decision(decision: str) -> None:
        row = current_row()
        key = (row["split"], row["image_id"])
        old_decision = decision_map.get(key, "unreviewed")
        old_reviewed_at = reviewed_at_map.get(key, pd.NA)
        if decision in QUOTAS[state["split"]]:
            count = counts_for_split(state["split"])[decision]
            if count >= QUOTAS[state["split"]][decision] and old_decision != decision:
                return
        undo_stack.append((key[0], key[1], old_decision, old_reviewed_at))
        decision_map[key] = decision
        reviewed_at_map[key] = pd.Timestamp.now(tz="UTC").isoformat()
        save_progress()
        move_next()

    def change_split(change) -> None:
        if change["name"] != "value":
            return
        state["split"] = change["new"]
        frame = split_candidates()
        first_unreviewed = next(
            (
                index for index, row in frame.iterrows()
                if decision_map.get((row["split"], row["image_id"]), "unreviewed") == "unreviewed"
            ),
            0,
        )
        state["index"] = first_unreviewed
        render()

    def previous(_button=None) -> None:
        state["index"] = max(0, state["index"] - 1)
        render()

    def undo(_button=None) -> None:
        if not undo_stack:
            return
        split, image_id, old_decision, old_reviewed_at = undo_stack.pop()
        key = (split, image_id)
        decision_map[key] = old_decision
        reviewed_at_map[key] = old_reviewed_at
        save_progress()
        state["split"] = split
        split_dropdown.value = split
        frame = split_candidates()
        matches = frame.index[frame["image_id"] == image_id].tolist()
        state["index"] = matches[0] if matches else 0
        render()

    split_dropdown.observe(change_split, names="value")
    crop_toggle.observe(lambda _change: render(), names="value")
    clear_button.on_click(lambda _button: record_decision("clear"))
    challenging_button.on_click(lambda _button: record_decision("challenging"))
    negative_button.on_click(lambda _button: record_decision("negative"))
    reject_button.on_click(lambda _button: record_decision("reject"))
    skip_button.on_click(lambda _button: record_decision("skip"))
    previous_button.on_click(previous)
    undo_button.on_click(undo)

    display(
        widgets.VBox(
            [
                split_dropdown,
                status,
                progress,
                metadata,
                crop_toggle,
                widgets.HBox([clear_button, challenging_button, negative_button]),
                widgets.HBox([reject_button, skip_button, previous_button, undo_button]),
                image_widget,
                crop_widget,
            ]
        )
    )
    render()

In [8]:
# Deliberately not launched automatically. Run this call when ready to review:
launch_review_widget()

## 6. Validate and materialize the selected dataset

Finalization is intentionally separate from review. It checks exact category quotas, source-image existence, duplicate IDs, and official split membership before writing `dataset_manifest.csv` or copying anything.

Selected images are copied to `data/dataset_v1/train/`, `val/`, and `test/`. Only generated image files in those three directories are synchronized; raw BDD100K files are never changed. Test images remain test-only and are not used for training or hyperparameter tuning.

In [9]:
ACCEPTED_DECISIONS = {"clear", "challenging", "negative"}


def validate_candidate_pool(candidates: pd.DataFrame) -> None:
    required = {
        "split", "image_id", "source_path", "vehicle_count", "max_vehicle_area",
        "max_vehicle_width", "max_vehicle_height", "candidate_type", "candidate_order",
    }
    missing_columns = required - set(candidates.columns)
    if missing_columns:
        raise ValueError(f"Candidate file is missing columns: {sorted(missing_columns)}")
    if candidates.duplicated(["split", "image_id"]).any():
        raise ValueError("Duplicate (split, image_id) pairs exist in candidates.csv.")
    if candidates["image_id"].duplicated().any():
        raise ValueError("An image ID appears in more than one candidate split.")
    if not set(candidates["split"]).issubset(SPLIT_IMAGE_DIRS):
        raise ValueError("An unknown split exists in candidates.csv.")

    for row in candidates.itertuples(index=False):
        source = (PROJECT_ROOT / row.source_path).resolve()
        expected_parent = SPLIT_IMAGE_DIRS[row.split].resolve()
        if source.parent != expected_parent:
            raise ValueError(
                f"Split-boundary violation: {row.image_id} is labelled {row.split} "
                f"but resolves under {source.parent}."
            )
        if not source.is_file():
            raise FileNotFoundError(f"Missing candidate source image: {source}")


def build_validated_manifest() -> pd.DataFrame:
    candidates = pd.read_csv(CANDIDATES_PATH)
    progress = pd.read_csv(REVIEW_PROGRESS_PATH, dtype="string")
    validate_candidate_pool(candidates)

    if progress.duplicated(["split", "image_id"]).any():
        raise ValueError("Duplicate (split, image_id) pairs exist in review_progress.csv.")
    selected = progress.loc[progress["decision"].isin(ACCEPTED_DECISIONS)].copy()
    selected = selected.merge(candidates, on=["split", "image_id"], how="left", validate="one_to_one")

    for split, expected in QUOTAS.items():
        actual = selected.loc[selected["split"] == split, "decision"].value_counts().to_dict()
        for category, quota in expected.items():
            if actual.get(category, 0) != quota:
                raise ValueError(
                    f"Quota not met for {split}/{category}: "
                    f"expected {quota}, found {actual.get(category, 0)}."
                )

    expected_total = sum(sum(values.values()) for values in QUOTAS.values())
    if len(selected) != expected_total:
        raise ValueError(f"Expected {expected_total} selected images, found {len(selected)}.")
    if selected["image_id"].duplicated().any():
        duplicates = selected.loc[selected["image_id"].duplicated(False), "image_id"].tolist()
        raise ValueError(f"Duplicate image IDs across final splits: {duplicates[:5]}")

    manifest_columns = [
        "image_id", "split", "decision", "source_path", "vehicle_count",
        "max_vehicle_area", "max_vehicle_width", "max_vehicle_height",
        "timeofday", "weather", "scene", "stats_source", "reviewed_at_utc",
    ]
    manifest = selected[manifest_columns].rename(columns={"decision": "sampling_category"})
    manifest = manifest.sort_values(["split", "sampling_category", "image_id"]).reset_index(drop=True)
    return manifest


def materialize_dataset(manifest: pd.DataFrame) -> None:
    "Synchronize only the generated split directories with the validated manifest."
    for split in SPLIT_IMAGE_DIRS:
        destination_dir = OUTPUT_DIR / split
        destination_dir.mkdir(parents=True, exist_ok=True)
        split_manifest = manifest.loc[manifest["split"] == split]
        expected_names = set(split_manifest["image_id"])

        for existing in destination_dir.iterdir():
            if existing.is_file() and existing.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                if existing.name not in expected_names:
                    existing.unlink()

        for row in split_manifest.itertuples(index=False):
            source = PROJECT_ROOT / row.source_path
            shutil.copy2(source, destination_dir / row.image_id)


def finalize_dataset() -> pd.DataFrame:
    manifest = build_validated_manifest()
    atomic_write_csv(manifest, MANIFEST_PATH)
    materialize_dataset(manifest)

    copied_counts = {
        split: len([path for path in (OUTPUT_DIR / split).iterdir() if path.is_file()])
        for split in SPLIT_IMAGE_DIRS
    }
    expected_counts = {split: sum(quotas.values()) for split, quotas in QUOTAS.items()}
    if copied_counts != expected_counts:
        raise RuntimeError(f"Copied image counts {copied_counts} do not match {expected_counts}.")

    print(f"Wrote {len(manifest)} rows to {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
    display(
        manifest.groupby(["split", "sampling_category"]).size().rename("count").to_frame()
    )
    return manifest

In [10]:
# Deliberately not finalized automatically. Run only after review quotas are complete:
dataset_manifest = finalize_dataset()

Wrote 200 rows to data\dataset_v1\dataset_manifest.csv


count
split sampling_category       
test  challenging            2
      clear                 16
      negative               2
train challenging           16
      clear                128
      negative              16
val   challenging            2
      clear                 16
      negative               2

## Outputs and next step

After the workflow is complete, `data/dataset_v1/` contains:

- `candidates.csv` — append-only review candidates and image-level statistics;
- `review_progress.csv` — immediately persisted review decisions;
- `dataset_manifest.csv` — the validated 200-image selection;
- `test_vehicle_stats.csv` — cached test-split COCO YOLO11n statistics; and
- `train/`, `val/`, and `test/` — copies of selected images.

The next milestone is licence plate annotation. Do not train a detector until dataset review, manifest validation, annotation guidelines, and annotation/export are complete.